# Семантический матчинг с помощью библиотеки sentence_transformers

В этом тюториале мы научимся семантическому поиску и ранжированию документов с помощью библиотеки _sentence_transformers_.

Документация библиотеки: https://www.sbert.net/

Ее можно поставить в виртуальное окружение командой `pip install sentence-transformers`

Импортируем модули которые нам понадобятся впоследствии:

In [1]:
import logging
from timeit import default_timer as timer

import numpy as np
import torch
import sentence_transformers as sbert

Подготовим логгер:

In [2]:
logging.basicConfig(level=logging.INFO)

## Данные

Будем работать с мини-сэмплом из пассажной части стандарного поискового датасета _MS MARCO_.

MS MARCO содержит:
- большую коллекцию документов, по которым можно искать
- реальные поисковые запросы из логов поисковика bing
- информацию о том, какие документы релевантны данному запросу

Подробнее про _MS MARCO_: https://microsoft.github.io/msmarco/

Возьмем несколько случайных запросов:

In [3]:
queries = [
    "how long is jupiter day",
    "what are isotopes of hydrogen",
]
queries

['how long is jupiter day', 'what are isotopes of hydrogen']

И несколько документов.

Выберем документы так, чтобы в нашем сэмпле были как релевантные запросам, так и нерелевантные, но "сложные": или содержащие ключевые слова, или просто близкие по тематике.

In [4]:
docs = [
    # About Jupiter
    "A day on Jupiter, also known as the sidereal rotation period, lasts 9.92496 hours.", # relevant
    "Voyager 2 arrived at Jupiter in 1979 two years after launching from Cape Canaveral",
    "It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium.",
    "The gas giant Jupiter has 63 known moons.",
    "Jupiter, the solar system s largest planet, takes 4,331.572 days to orbit around the sun",
    "Jupiter orbits the Sun every 11.86 Earth years (or 4,332 days)",
    "Given the fact that it is the largest planet in the Solar System, one would expect that a day on Jupiter would last a long time.",
    "Io takes 1.77 Earth days to orbit Jupiter.",

    # About hydrogen
    "Hydrogen has three isotopes: hydrogen-1 (protium), hydrogen-2 (deuterium) and hydrogen-3 (tritium).", # relevant
    "A deuterium atom contains one proton, one neutron, and one electron.",
    "The fusion of deuterium with tritium creates helium-4, frees a remaining neutron, and releases energy.",
    "These different versions of hydrogen are called isotopes.",
    "Hydrogen is the only element where the isotopes are given specific names.",
    "The 2 H (or hydrogen-2) isotope is usually called deuterium",
    "Tritium (hydrogen-3 or 3H) is a weakly radioactive isotope of the element hydrogen",
    "All elements have at least some radioactive isotopes.",
    
    # Misc
    "The official language in Bangladesh is Bengali, though English is widely spoken.",
    "Grasshoppers are medium to large insects.",
    "Debian and Ubuntu are the most influential Linux distributions ever.",
    "Мoscow is the largest city in Russia with a population of 12.1 million people",
]

docs

['A day on Jupiter, also known as the sidereal rotation period, lasts 9.92496 hours.',
 'Voyager 2 arrived at Jupiter in 1979 two years after launching from Cape Canaveral',
 "It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium.",
 'The gas giant Jupiter has 63 known moons.',
 'Jupiter, the solar system s largest planet, takes 4,331.572 days to orbit around the sun',
 'Jupiter orbits the Sun every 11.86 Earth years (or 4,332 days)',
 'Given the fact that it is the largest planet in the Solar System, one would expect that a day on Jupiter would last a long time.',
 'Io takes 1.77 Earth days to orbit Jupiter.',
 'Hydrogen has three isotopes: hydrogen-1 (protium), hydrogen-2 (deuterium) and hydrogen-3 (tritium).',
 'A deuterium atom contains one proton, one neutron, and one electron.',
 'The fusion of deuterium with tritium creates helium-4, frees a remaining neutron, and releases energy.',
 'These different versions of hydrogen are call

## Семантический поиск

Начнем с того, что попробуем поискать наши запросы в нашей коллекции документов.

Для этого нам надо:
- заэмбеддить все документы
- заэмбеддить запросы
- найти к каждому запросу ближайшие документы по косинусному расстонию

Для простоты мы не будем использовать какой-либо векторный индекс (типа _faiss_), и вместо приближенного поиска ближайших соседей (ANN) будем делать полный KNN.

Подготовим устройство:

In [5]:
device = torch.device("cuda:0")
logging.info(f"created device {device}")

INFO:root:created device cuda:0


Загрузим модели семантического поиска/матчинга.

Таких моделей существует множество, они могут отличаться:
- по качеству
- размеру и скорости
- поддержке разных языков и т.п.

Для выбора моделей рекомендую опираться на бенчмарки, напр. на MTEB:
- тут можно найти описание и ссылку на лидерборд: https://github.com/embeddings-benchmark/mteb
- немного про сам бенчмарк: https://huggingface.co/blog/mteb

Обычно, для русского языка в 2026 г. используют какую-либо модель из:
- семейства моделей E5 от Microsoft, напр. _multilingual-e5-large_
- новых Qwen3-Embedding, напр. _Qwen3-Embedding-8B_
- моделей BGE, напр. _bge-m3_
- моделей от Сбера, напр. _FRIDA_

Мы с вами дальше будем работать с английским языком, поэтому нам подойдут более легкие модели из семейства _MiniLM_.

In [6]:
# Which model?
model_name = "sentence-transformers/all-MiniLM-L6-v2"
#model_name = "intfloat/multilingual-e5-base"
#model_name = "Qwen/Qwen3-Embedding-0.6B" # requires prompt!

# Model params
model_kwargs = {}
model_kwargs['dtype'] = torch.float16

# Load model
model = sbert.SentenceTransformer(model_name, device=str(device), trust_remote_code=True, model_kwargs=model_kwargs)
num_params = sum(t.numel() for t in model.parameters())
logging.info(f"loaded model: model_name = {model_name} num_params = {num_params}")

INFO:httpx:HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/2_Normalize/config.json "HTTP/1.1 404 Not Found"
INFO:root:loaded model: model_name = sentence-transformers/all-MiniLM-L6-v2 num_params = 22713216


Убедимся, что модель загрузилась на GPU с помощью утилиты _nvidia-smi_:

In [8]:
!nvidia-smi

Sat Nov  7 03:23:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 620.70.06              KMD Version: 620.70.06     CUDA UMD Version: 13.5     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 Ti     On  |   00000000:01:00.0 Off |                  N/A |
|  0%   35C    P8              1W /  165W |    8975MiB /  16380MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Заэмбеддим запросы:

In [10]:
start = timer()
query_embeds = model.encode(queries)
logging.info(f"embedded queries: query_embeds.shape = {query_embeds.shape} elapsed = {timer() - start:.3f}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:embedded queries: query_embeds.shape = (2, 384) elapsed = 0.020


И, аналогично, заэмбеддим документы:

In [11]:
start = timer()
doc_embeds = model.encode(docs)
logging.info(f"embedded docs: doc_embeds.shape = {doc_embeds.shape} elapsed = {timer() - start:.3f}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:embedded docs: doc_embeds.shape = (20, 384) elapsed = 0.014


Посчитаем скоры близости, как правило это просто косинус между векторами-эмбеддингами:

In [12]:
scores = model.similarity(query_embeds, doc_embeds).cpu().numpy()
print(scores)
logging.info(f"computed similarities: scores.shape = {scores.shape}")

INFO:root:computed similarities: scores.shape = (2, 20)


[[ 7.59093881e-01  4.17146146e-01  4.59765553e-01  5.21613955e-01
   6.79894984e-01  7.31379092e-01  7.68550277e-01  7.17413664e-01
   5.52056544e-02  1.26533285e-02 -1.28006451e-02  2.83714160e-02
   9.99835879e-03  3.66813354e-02  1.25497691e-02 -4.19245027e-02
   3.22293397e-03  5.50573245e-02 -8.57039019e-02 -6.13156483e-02]
 [ 6.43135458e-02  1.53208664e-02  2.85269380e-01  1.16806239e-01
   1.07255816e-01  1.95189565e-01  2.12066397e-02  8.55834112e-02
   8.12239408e-01  5.40480018e-01  4.19135928e-01  8.59364986e-01
   7.91978002e-01  7.50021696e-01  7.21943617e-01  4.85067606e-01
   4.13253903e-03  1.20238654e-01  4.74446751e-02  6.68793917e-04]]


Напишем функцию, которая ранжирует документы по запросу, и возвращает ранжированный список пар документ-скор:

In [13]:
def rank_docs(docs, query_scores):
    ranked_doc_ids = np.argsort(query_scores)[::-1]
    ranked_docs = [docs[i] for i in ranked_doc_ids]
    doc_scores = query_scores[ranked_doc_ids].tolist()
    docs_with_scores = list(zip(ranked_docs, doc_scores))
    return docs_with_scores

Применим к первому запросу про Юпитер:

In [15]:
print(f"Query: {queries[0]}")
rank_docs(docs, scores[0])

Query: how long is jupiter day


[('Given the fact that it is the largest planet in the Solar System, one would expect that a day on Jupiter would last a long time.',
  0.7685502767562866),
 ('A day on Jupiter, also known as the sidereal rotation period, lasts 9.92496 hours.',
  0.7590938806533813),
 ('Jupiter orbits the Sun every 11.86 Earth years (or 4,332 days)',
  0.7313790917396545),
 ('Io takes 1.77 Earth days to orbit Jupiter.', 0.7174136638641357),
 ('Jupiter, the solar system s largest planet, takes 4,331.572 days to orbit around the sun',
  0.6798949837684631),
 ('The gas giant Jupiter has 63 known moons.', 0.5216139554977417),
 ("It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium.",
  0.45976555347442627),
 ('Voyager 2 arrived at Jupiter in 1979 two years after launching from Cape Canaveral',
  0.41714614629745483),
 ('Hydrogen has three isotopes: hydrogen-1 (protium), hydrogen-2 (deuterium) and hydrogen-3 (tritium).',
  0.055205654352903366),
 ('Grasshop

Видим, что ранжирование по близости эмбеддингов работает неплохо, но не идеально.

С одной стороны, документы про Юпитер действительно поднялись выше, что документы про водород. С другой стороны, на первое место вылез документ, содержащий все ключевые слова, но не отвечающий на вопрос пользователя.

Посмотрим теперь как ранжируются документы по второму запросу про водород:

In [16]:
print(f"Query: {queries[1]}")
rank_docs(docs, scores[1])

Query: what are isotopes of hydrogen


[('These different versions of hydrogen are called isotopes.',
  0.8593649864196777),
 ('Hydrogen has three isotopes: hydrogen-1 (protium), hydrogen-2 (deuterium) and hydrogen-3 (tritium).',
  0.812239408493042),
 ('Hydrogen is the only element where the isotopes are given specific names.',
  0.7919780015945435),
 ('The 2 H (or hydrogen-2) isotope is usually called deuterium',
  0.7500216960906982),
 ('Tritium (hydrogen-3 or 3H) is a weakly radioactive isotope of the element hydrogen',
  0.7219436168670654),
 ('A deuterium atom contains one proton, one neutron, and one electron.',
  0.5404800176620483),
 ('All elements have at least some radioactive isotopes.',
  0.48506760597229004),
 ('The fusion of deuterium with tritium creates helium-4, frees a remaining neutron, and releases energy.',
  0.4191359281539917),
 ("It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium.",
  0.2852693796157837),
 ('Jupiter orbits the Sun every 11.86 Eart

Тут у нас аналогичная картина -- ранжирование хорошее, но не идеальное.

## Ранжирование

Попробуем немного исправить ситуацию и улучшить качество ранжирования за счет использования модели-реранкера.

На всякий случай вспомним:
- модели-эмбеддеры (их еще называют _би-энкодеры_) получают на вход по отдельности текст документа или запроса, и выдают эмбеддинги
- модели-реранкеры (_кросс-энкодеры_) получают на вход сразу два текста: и запроса, и документа, и выдают число (ранк), по которому можно ранжировать документы

Главный плюс би-энкодеров: эмбеддинги документов можно предрасчитать заранее и сохранить в векторном индексе

Главные минусы кросс-энкодеров:
- ничего предрасчитать заранее не получится
- надо применять ко всем парам запрос-документ -- это медленно!
- как правило, получается переранжировать только небольшой топ

Но у кросс-энкодеров есть и плюс: ранжирование получается гораздо более точным.

Загрузим модели, тут тоже будем использовать легкую модель _MiniLM_ обученную на датасете MS MARCO:

In [17]:
# Which model?
ce_model_name = "cross-encoder/ms-marco-MiniLM-L6-v2"
#ce_model_name = "Qwen/Qwen3-Reranker-0.6B"

# Load model
ce_model = sbert.CrossEncoder(ce_model_name, device=str(device), model_kwargs=model_kwargs)
logging.info(f"loaded cross-encoder model: ce_model_name = {ce_model_name}")

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/233902d25c440f23af6f7d6e94d2946bac0bee0a/modules.json "HTTP/1.1 404 Not Found"
INFO:sentence_transformers.base.model:No modules.json found for cross-encoder/ms-marco-MiniLM-L6-v2, initializing a new CrossEncoder model.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:root:loaded cross-encoder model: ce_model_name = cross-encoder/ms-marco-MiniLM-L6-v2


Представим данные в виде пар запрос-документ, т.к. в этом формате их ожидает модель:

In [18]:
num_docs = len(docs)

# Create query/doc pairs
q0 = [queries[0]] * num_docs
pairs0 = list(zip(q0, docs))
pairs0

[('how long is jupiter day',
  'A day on Jupiter, also known as the sidereal rotation period, lasts 9.92496 hours.'),
 ('how long is jupiter day',
  'Voyager 2 arrived at Jupiter in 1979 two years after launching from Cape Canaveral'),
 ('how long is jupiter day',
  "It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium."),
 ('how long is jupiter day', 'The gas giant Jupiter has 63 known moons.'),
 ('how long is jupiter day',
  'Jupiter, the solar system s largest planet, takes 4,331.572 days to orbit around the sun'),
 ('how long is jupiter day',
  'Jupiter orbits the Sun every 11.86 Earth years (or 4,332 days)'),
 ('how long is jupiter day',
  'Given the fact that it is the largest planet in the Solar System, one would expect that a day on Jupiter would last a long time.'),
 ('how long is jupiter day', 'Io takes 1.77 Earth days to orbit Jupiter.'),
 ('how long is jupiter day',
  'Hydrogen has three isotopes: hydrogen-1 (protium), hydr

Аналогично для второго запроса:

In [19]:
q1 = [queries[1]] * num_docs
pairs1 = list(zip(q1, docs))
pairs1

[('what are isotopes of hydrogen',
  'A day on Jupiter, also known as the sidereal rotation period, lasts 9.92496 hours.'),
 ('what are isotopes of hydrogen',
  'Voyager 2 arrived at Jupiter in 1979 two years after launching from Cape Canaveral'),
 ('what are isotopes of hydrogen',
  "It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium."),
 ('what are isotopes of hydrogen',
  'The gas giant Jupiter has 63 known moons.'),
 ('what are isotopes of hydrogen',
  'Jupiter, the solar system s largest planet, takes 4,331.572 days to orbit around the sun'),
 ('what are isotopes of hydrogen',
  'Jupiter orbits the Sun every 11.86 Earth years (or 4,332 days)'),
 ('what are isotopes of hydrogen',
  'Given the fact that it is the largest planet in the Solar System, one would expect that a day on Jupiter would last a long time.'),
 ('what are isotopes of hydrogen',
  'Io takes 1.77 Earth days to orbit Jupiter.'),
 ('what are isotopes of hydrogen',


Применим наши кросс-энкодеры к парам запрос-документ:

In [20]:
start = timer()
scores0 = ce_model.predict(pairs0)
print(scores0)
logging.info(f"reranked: query = {queries[0]} elapsed = {timer() - start:.3f}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:reranked: query = how long is jupiter day elapsed = 0.030


[  8.7109375  -5.1914062  -9.7890625  -7.4453125   5.7382812   6.0117188
   2.8652344   3.1933594 -11.2109375 -11.28125   -11.2265625 -11.34375
 -11.34375   -11.2890625 -11.15625   -11.328125  -11.2734375 -11.3046875
 -11.359375  -11.359375 ]


Аналогично и для второго запроса:

In [21]:
start = timer()
scores1 = ce_model.predict(pairs1)
print(scores1)
logging.info(f"reranked: query = {queries[1]} elapsed = {timer() - start:.3f}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:reranked: query = what are isotopes of hydrogen elapsed = 0.025


[-11.234375  -11.296875  -10.109375  -11.3828125 -11.2890625 -11.2734375
 -11.3515625 -11.34375     8.5625     -8.6953125  -9.703125    5.2148438
   4.0273438   4.6289062   4.7695312  -5.5351562 -11.2734375 -11.3125
 -11.375     -11.265625 ]


Посмотрим что получилось, сначала для первого запроса про Юпитер:

In [22]:
print(f"Query: {queries[0]}")
rank_docs(docs, scores0)

Query: how long is jupiter day


[('A day on Jupiter, also known as the sidereal rotation period, lasts 9.92496 hours.',
  8.7109375),
 ('Jupiter orbits the Sun every 11.86 Earth years (or 4,332 days)',
  6.01171875),
 ('Jupiter, the solar system s largest planet, takes 4,331.572 days to orbit around the sun',
  5.73828125),
 ('Io takes 1.77 Earth days to orbit Jupiter.', 3.193359375),
 ('Given the fact that it is the largest planet in the Solar System, one would expect that a day on Jupiter would last a long time.',
  2.865234375),
 ('Voyager 2 arrived at Jupiter in 1979 two years after launching from Cape Canaveral',
  -5.19140625),
 ('The gas giant Jupiter has 63 known moons.', -7.4453125),
 ("It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium.",
  -9.7890625),
 ('Tritium (hydrogen-3 or 3H) is a weakly radioactive isotope of the element hydrogen',
  -11.15625),
 ('Hydrogen has three isotopes: hydrogen-1 (protium), hydrogen-2 (deuterium) and hydrogen-3 (tritium).'

Результат стал гораздо лучше!

Теперь документы не просто разделились на релевантные и совсем не релевантные, но и внутри группы релевантных более релевантные оказались выше, чем менее релевантные.

И, аналогично, для второго:

In [23]:
print(f"Query: {queries[1]}")
rank_docs(docs, scores1)

Query: what are isotopes of hydrogen


[('Hydrogen has three isotopes: hydrogen-1 (protium), hydrogen-2 (deuterium) and hydrogen-3 (tritium).',
  8.5625),
 ('These different versions of hydrogen are called isotopes.', 5.21484375),
 ('Tritium (hydrogen-3 or 3H) is a weakly radioactive isotope of the element hydrogen',
  4.76953125),
 ('The 2 H (or hydrogen-2) isotope is usually called deuterium', 4.62890625),
 ('Hydrogen is the only element where the isotopes are given specific names.',
  4.02734375),
 ('All elements have at least some radioactive isotopes.', -5.53515625),
 ('A deuterium atom contains one proton, one neutron, and one electron.',
  -8.6953125),
 ('The fusion of deuterium with tritium creates helium-4, frees a remaining neutron, and releases energy.',
  -9.703125),
 ("It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium.",
  -10.109375),
 ('A day on Jupiter, also known as the sidereal rotation period, lasts 9.92496 hours.',
  -11.234375),
 ('Мoscow is the larg

Обратим внимание на значения ранков -- это больше не косинусное расстояние, а некая величина, не имеющая физического смысла.

Распространенная ошибка: подбирать какой-то порог на значение ранка, и фильровать по нему документы -- так делать нельзя!

Таким образом, мы научились:
- пользоваться библиотекой _sentence_transformers_
- эмбеддить запросы и документы с помощью моделей-эмбеддеров
- считать скоры семантической близости
- переранжировать документы с помощью моделей кросс-энкодеров

Дальше для самостоятельного изучения рекомендуем:
- попробовать другие модели
- взять датасет побольше, напр. полный _MS MARCO_
- научиться считать метрики качества ранжирования и сравнить разные модели между собой